# 路线偏好训练数据诊断

这个 notebook 用于判断当前路线偏好 MLP 是应该继续调参，还是优先补数据。

- 只读 PostgreSQL 和训练产物，不写数据库。
- 复用 `training/` 里的数据构造、candidate_set 切分和模型定义，避免诊断口径和训练口径不一致。
- 图表重点看 candidate set 数量、pair 数量、accepted/rejected 分布、reason code 支持度、训练 history，以及当前模型的 reason 分数分布。


In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import torch
from IPython.display import Markdown, display

try:
    import pandas as pd
except ImportError:
    pd = None


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in (current, *current.parents):
        if (path / "AGENTS.md").exists() and (path / "ai-python" / "src").exists():
            return path
    raise RuntimeError("找不到项目根目录，请从 urban-sidequest 项目内打开 notebook。")


PROJECT_ROOT = find_project_root()
SRC_ROOT = PROJECT_ROOT / "ai-python" / "src"
SRC_ROOT_TEXT = str(SRC_ROOT)
sys.path = [SRC_ROOT_TEXT] + [entry for entry in sys.path if entry != SRC_ROOT_TEXT]
for module_name in list(sys.modules):
    if module_name == "urban_sidequest_ai" or module_name.startswith("urban_sidequest_ai."):
        del sys.modules[module_name]

OUTPUT_DIR = PROJECT_ROOT / "tmp" / "route-pref-training-output"
FEATURE_SCHEMA_VERSION = "route_pref_v5"
DEFAULT_NOTEBOOK_TRAIN_SEED = 23
DEFAULT_NOTEBOOK_SPLIT_SEED = 13


def load_training_seed_context(output_dir: Path) -> tuple[int, int]:
    history_path = output_dir / "history.jsonl"
    if not history_path.exists():
        return DEFAULT_NOTEBOOK_TRAIN_SEED, DEFAULT_NOTEBOOK_SPLIT_SEED
    for line in history_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        record = json.loads(line)
        return int(record.get("seed", DEFAULT_NOTEBOOK_TRAIN_SEED)), int(record.get("splitSeed", DEFAULT_NOTEBOOK_SPLIT_SEED))
    return DEFAULT_NOTEBOOK_TRAIN_SEED, DEFAULT_NOTEBOOK_SPLIT_SEED


TRAIN_SEED, SPLIT_SEED = load_training_seed_context(OUTPUT_DIR)
SEED = TRAIN_SEED

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25


def show_table(rows: list[dict], title: str | None = None, sort_by: str | None = None):
    if title:
        display(Markdown(f"### {title}"))
    if not rows:
        display(Markdown("无数据"))
        return None
    if pd is not None:
        frame = pd.DataFrame(rows)
        if sort_by and sort_by in frame.columns:
            frame = frame.sort_values(sort_by, ascending=False)
        display(frame)
        return frame
    for row in rows:
        print(row)
    return rows


display(Markdown(f"项目根目录：`{PROJECT_ROOT}`"))
display(Markdown(f"训练产物目录：`{OUTPUT_DIR}`"))
display(Markdown(f"诊断切分：train seed `{TRAIN_SEED}`，split seed `{SPLIT_SEED}`"))


In [ ]:
from urban_sidequest_ai.models.route_preference.training.db import connect, load_database_config
from urban_sidequest_ai.models.route_preference.training.dataset import (
    build_dataset_bundle,
    iter_batches,
    split_by_candidate_set,
)
from urban_sidequest_ai.models.route_preference.training.model import RoutePreferenceModel, RoutePreferenceModelConfig
from urban_sidequest_ai.models.route_preference.training.repository import RoutePreferenceTrainingRepository
from urban_sidequest_ai.models.route_preference.training.schema import (
    DEFAULT_GOOD_ROUTE_THRESHOLD,
    DEFAULT_HIGH_ISSUE_THRESHOLD,
    DEFAULT_ISSUE_THRESHOLD,
    InvalidJudgmentPolicy,
    REASON_CODES,
)


def load_bundle():
    db_config = load_database_config()
    with connect(db_config) as connection:
        repository = RoutePreferenceTrainingRepository(connection)
        sample_rows = repository.fetch_training_samples(FEATURE_SCHEMA_VERSION)
        judgment_rows = repository.fetch_completed_judgments({row.candidate_set_id for row in sample_rows})
    bundle = build_dataset_bundle(sample_rows, judgment_rows, InvalidJudgmentPolicy.FAIL)
    splits = split_by_candidate_set(bundle.groups, seed=SPLIT_SEED)
    return sample_rows, judgment_rows, bundle, splits


sample_rows, judgment_rows, bundle, splits = load_bundle()
SPLIT_GROUPS = {
    "train": splits.train,
    "valid": splits.valid,
    "test": splits.test,
}

display(Markdown(f"读取训练样本 `{len(sample_rows)}` 行，completed judgments `{len(judgment_rows)}` 条，可训练 candidate sets `{len(bundle.groups)}` 组。"))
if bundle.skipped_judgments:
    display(Markdown(f"跳过 judgment 数：`{len(bundle.skipped_judgments)}`"))
    for message in bundle.skipped_judgments[:10]:
        print(message)


In [ ]:
def summarize_split(groups):
    route_count = sum(len(group.items) for group in groups)
    pair_count = sum(len(group.pairs) for group in groups)
    accepted_count = sum(1 for group in groups for item in group.items if item.is_accepted)
    rejected_count = sum(1 for group in groups for item in group.items if item.is_rejected)
    goodness_labeled = sum(1 for group in groups for item in group.items if item.goodness_mask)
    reason_labeled = sum(1 for group in groups for item in group.items if item.reason_mask)
    return {
        "candidate_sets": len(groups),
        "routes": route_count,
        "pairs": pair_count,
        "avg_routes_per_set": round(route_count / len(groups), 2) if groups else 0.0,
        "avg_pairs_per_set": round(pair_count / len(groups), 2) if groups else 0.0,
        "accepted_routes": accepted_count,
        "rejected_routes": rejected_count,
        "goodness_labeled_routes": goodness_labeled,
        "reason_labeled_routes": reason_labeled,
    }


split_rows = [{"split": split_name, **summarize_split(groups)} for split_name, groups in SPLIT_GROUPS.items()]
show_table(split_rows, "Split 总览")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
x = [row["split"] for row in split_rows]
axes[0].bar(x, [row["candidate_sets"] for row in split_rows])
axes[0].set_title("Candidate sets")
axes[1].bar(x, [row["routes"] for row in split_rows])
axes[1].set_title("Routes")
axes[2].bar(x, [row["pairs"] for row in split_rows])
axes[2].set_title("Pairs")
plt.tight_layout()
plt.show()


In [ ]:
def distribution_rows(counter: Counter, key_name: str, split_name: str):
    total = sum(counter.values())
    return [
        {
            "split": split_name,
            key_name: key,
            "count": count,
            "pct": round(count / total, 4) if total else 0.0,
        }
        for key, count in counter.items()
    ]


judge_rows = []
for split_name, groups in SPLIT_GROUPS.items():
    judge_rows.extend(distribution_rows(Counter(group.judge_type for group in groups), "judge_type", split_name))

show_table(judge_rows, "judge_type 分布", "count")


In [ ]:
reason_rows = []
for split_name, groups in SPLIT_GROUPS.items():
    reason_route_count = sum(1 for group in groups for item in group.items if item.reason_mask)
    rejected_count = sum(1 for group in groups for item in group.items if item.is_rejected)
    positives = Counter()
    cooccurrence = Counter()
    for group in groups:
        for item in group.items:
            if not item.reason_mask:
                continue
            active_codes = [REASON_CODES[index] for index, value in enumerate(item.reason_labels) if value >= 0.5]
            positives.update(active_codes)
            cooccurrence[len(active_codes)] += 1
    for code in REASON_CODES:
        count = positives[code]
        reason_rows.append(
            {
                "split": split_name,
                "reason_code": code,
                "positive_routes": count,
                "positive_rate_in_reason_routes": round(count / reason_route_count, 4) if reason_route_count else 0.0,
                "positive_rate_in_rejected": round(count / rejected_count, 4) if rejected_count else 0.0,
            }
        )
    show_table(
        [{"split": split_name, "active_reason_count_per_route": key, "routes": value} for key, value in sorted(cooccurrence.items())],
        f"{split_name} 每条 rejected route 的 reason 数量",
    )

reason_frame = show_table(reason_rows, "reason code 正样本支持度", "positive_routes")

if pd is not None:
    pivot = reason_frame.pivot(index="reason_code", columns="split", values="positive_routes").fillna(0)
    pivot.loc[list(REASON_CODES)].plot(kind="bar", figsize=(14, 5))
    plt.title("Reason positive routes by split")
    plt.ylabel("positive route count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

low_support = [row for row in reason_rows if row["split"] == "train" and row["positive_routes"] < 30]
if low_support:
    display(Markdown("### 训练集低支持度 reason code（正样本 < 30）"))
    show_table(low_support, sort_by="positive_routes")


In [ ]:
pair_rows = []
for split_name, groups in SPLIT_GROUPS.items():
    pair_type_counter = Counter(pair.pair_type for group in groups for pair in group.pairs)
    weights_by_type = defaultdict(list)
    for group in groups:
        for pair in group.pairs:
            weights_by_type[pair.pair_type].append(pair.weight_raw)
    for pair_type, count in pair_type_counter.items():
        weights = weights_by_type[pair_type]
        pair_rows.append(
            {
                "split": split_name,
                "pair_type": pair_type,
                "pairs": count,
                "weight_min": round(min(weights), 4),
                "weight_mean": round(sum(weights) / len(weights), 4),
                "weight_max": round(max(weights), 4),
            }
        )

pair_frame = show_table(pair_rows, "pair 类型与权重分布", "pairs")
if pd is not None:
    pair_frame.pivot(index="pair_type", columns="split", values="pairs").fillna(0).plot(kind="bar", figsize=(10, 4))
    plt.title("Pair counts by type")
    plt.ylabel("pair count")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


In [ ]:
history_path = OUTPUT_DIR / "history.jsonl"
if not history_path.exists():
    display(Markdown(f"没有找到训练历史：`{history_path}`"))
else:
    metric_history_rows = [json.loads(line) for line in history_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    metric_history_frame = pd.DataFrame(metric_history_rows) if pd is not None else None

    metric_groups = {
        "Rank": [
            ("ndcg@3", "valid/ndcg@3"),
            ("top1", "valid/top1Accuracy"),
            ("top2 hit", "valid/top2HitRate"),
            ("pairwise", "valid/pairwiseAccuracy"),
            ("weighted pairwise", "valid/weightedPairwiseAccuracy"),
        ],
        "Goodness": [
            ("accuracy@0.5", "valid/goodnessAccuracy@0.5"),
            ("AUC", "valid/goodnessAuc"),
            ("PR-AUC", "valid/goodnessPrAuc"),
        ],
        "Reason": [
            ("micro-F1@0.5", "valid/reasonConditionalMicroF1@0.5"),
            ("macro-F1@0.5", "valid/reasonConditionalMacroF1@0.5"),
            ("per-code AUC macro", "valid/reasonConditionalPerCodeAucMacro"),
            ("top issue hit", "valid/reasonTopIssueHitRate"),
        ],
    }
    loss_groups = {
        "Rank": [
            ("train ranking", "train/loss/ranking"),
            ("valid ranking", "valid/loss/ranking"),
        ],
        "Goodness": [
            ("train goodness", "train/loss/goodness"),
            ("valid goodness", "valid/loss/goodness"),
        ],
        "Reason": [
            ("train reason", "train/loss/reason"),
            ("valid reason", "valid/loss/reason"),
        ],
    }

    summary_rows = []
    for task_name, metrics in metric_groups.items():
        for label, metric_key in metrics:
            values = [row[metric_key] for row in metric_history_rows if metric_key in row]
            if not values:
                continue
            best_value = max(values)
            best_row = next(row for row in metric_history_rows if row.get(metric_key) == best_value)
            current_value = metric_history_rows[-1].get(metric_key)
            summary_rows.append(
                {
                    "task": task_name,
                    "metric": label,
                    "best_epoch": int(best_row["epoch"]),
                    "best_value": round(best_value, 4),
                    "current_epoch": int(metric_history_rows[-1]["epoch"]),
                    "current_value": round(current_value, 4) if current_value is not None else None,
                }
            )
    show_table(summary_rows, "Rank / Goodness / Reason 主指标总览")

    if metric_history_frame is not None:
        fig, axes = plt.subplots(3, 2, figsize=(15, 12), sharex=True)
        task_names = ["Rank", "Goodness", "Reason"]
        for row_index, task_name in enumerate(task_names):
            loss_axis = axes[row_index][0]
            metric_axis = axes[row_index][1]

            for label, metric_key in loss_groups[task_name]:
                if metric_key in metric_history_frame:
                    loss_axis.plot(metric_history_frame["epoch"], metric_history_frame[metric_key], marker="o", label=label)
            loss_axis.set_title(f"{task_name} loss")
            loss_axis.set_xlabel("epoch")
            loss_axis.legend()

            metric_values = []
            for label, metric_key in metric_groups[task_name]:
                if metric_key in metric_history_frame:
                    series = metric_history_frame[metric_key]
                    metric_values.extend(series.dropna().tolist())
                    metric_axis.plot(metric_history_frame["epoch"], series, marker="o", label=label)
            metric_axis.set_title(f"{task_name} metrics")
            metric_axis.set_xlabel("epoch")
            if metric_values:
                metric_min = min(metric_values)
                metric_max = max(metric_values)
                metric_span = max(metric_max - metric_min, 0.02)
                metric_padding = metric_span * 0.18
                metric_axis.set_ylim(
                    max(0.0, metric_min - metric_padding),
                    min(1.0, metric_max + metric_padding),
                )
            metric_axis.legend()
        plt.tight_layout()
        plt.show()

        per_code_rows = []
        for reason_code in REASON_CODES:
            metric_key = f"valid/reasonAuc/{reason_code}"
            if metric_key not in metric_history_frame:
                continue
            best_idx = metric_history_frame[metric_key].idxmax()
            per_code_rows.append(
                {
                    "reason_code": reason_code,
                    "best_epoch": int(metric_history_frame.loc[best_idx, "epoch"]),
                    "best_auc": round(metric_history_frame.loc[best_idx, metric_key], 4),
                    "current_auc": round(metric_history_frame.iloc[-1][metric_key], 4),
                }
            )
        show_table(per_code_rows, "Reason per-code AUC", "current_auc")


In [ ]:
def percentile(values: list[float], q: float) -> float:
    if not values:
        return 0.0
    ordered = sorted(values)
    pos = (len(ordered) - 1) * q
    lo = int(pos)
    hi = min(lo + 1, len(ordered) - 1)
    frac = pos - lo
    return ordered[lo] * (1 - frac) + ordered[hi] * frac


def model_config_from_checkpoint(payload: dict) -> RoutePreferenceModelConfig:
    config = payload["modelConfig"]
    return RoutePreferenceModelConfig(
        route_derived_dim=config["routeDerivedDim"],
        context_cross_dim=config["contextCrossDim"],
        intra_set_dim=config.get("intraSetDim", 0),
        stop_dim=config["stopDim"],
        segment_dim=config["segmentDim"],
        max_stops=config["maxStops"],
        max_segments=config["maxSegments"],
        hidden_dim=config["hiddenDim"],
        dropout=config["dropout"],
        reason_hidden_dim=config["reasonHiddenDim"],
        reason_count=config["reasonCount"],
    )


def current_model_reason_distribution(groups):
    checkpoint_path = OUTPUT_DIR / "route-preference.pt"
    if not checkpoint_path.exists():
        return None
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    model = RoutePreferenceModel(model_config_from_checkpoint(checkpoint))
    model.load_state_dict(checkpoint["stateDict"])
    model.eval()

    rows = []
    with torch.no_grad():
        for split_name, split_groups in groups.items():
            for group in split_groups:
                batch = next(iter_batches((group,), 1, shuffle=False, seed=SEED, device=torch.device("cpu")))
                output = model(
                    batch.stop_matrix,
                    batch.segment_matrix,
                    batch.route_derived_vector,
                    batch.context_cross_vector,
                    batch.intra_set_vector,
                )
                goodness_probs = torch.sigmoid(output.route_goodness_logit).cpu().tolist()
                reason_probs = torch.sigmoid(output.reason_code_logits).cpu().tolist()
                for index, item in enumerate(group.items):
                    max_reason = max(reason_probs[index]) if reason_probs[index] else 0.0
                    labeled_reason_scores = [reason_probs[index][code_index] for code_index, value in enumerate(item.reason_labels) if value >= 0.5]
                    runtime_threshold = DEFAULT_HIGH_ISSUE_THRESHOLD if goodness_probs[index] >= DEFAULT_GOOD_ROUTE_THRESHOLD else DEFAULT_ISSUE_THRESHOLD
                    rows.append(
                        {
                            "split": split_name,
                            "candidate_set_id": group.candidate_set_id,
                            "route_code": item.route_code,
                            "is_accepted": item.is_accepted,
                            "is_rejected": item.is_rejected,
                            "goodness_prob": goodness_probs[index],
                            "max_reason_prob": max_reason,
                            "max_labeled_reason_prob": max(labeled_reason_scores) if labeled_reason_scores else None,
                            "runtime_has_issue": max_reason >= runtime_threshold,
                        }
                    )
    return rows


score_rows = current_model_reason_distribution(SPLIT_GROUPS)
if score_rows is None:
    display(Markdown(f"没有找到当前模型：`{OUTPUT_DIR / 'route-preference.pt'}`"))
else:
    summary_rows = []
    for split_name in ["train", "valid", "test"]:
        accepted = [row for row in score_rows if row["split"] == split_name and row["is_accepted"]]
        rejected = [row for row in score_rows if row["split"] == split_name and row["is_rejected"]]
        accepted_max = [row["max_reason_prob"] for row in accepted]
        rejected_labeled = [row["max_labeled_reason_prob"] for row in rejected if row["max_labeled_reason_prob"] is not None]
        summary_rows.append(
            {
                "split": split_name,
                "accepted_routes": len(accepted),
                "accepted_runtime_issue_rate": round(sum(row["runtime_has_issue"] for row in accepted) / len(accepted), 4) if accepted else 0.0,
                "accepted_max_reason_p50": round(percentile(accepted_max, 0.50), 4),
                "accepted_max_reason_p90": round(percentile(accepted_max, 0.90), 4),
                "accepted_max_reason_>=0.5": round(sum(value >= 0.5 for value in accepted_max) / len(accepted_max), 4) if accepted_max else 0.0,
                "accepted_max_reason_>=0.8": round(sum(value >= 0.8 for value in accepted_max) / len(accepted_max), 4) if accepted_max else 0.0,
                "rejected_labeled_reason_p50": round(percentile(rejected_labeled, 0.50), 4),
                "rejected_labeled_reason_p90": round(percentile(rejected_labeled, 0.90), 4),
            }
        )
    show_table(summary_rows, "当前模型 reason 分数阈值诊断")

    if pd is not None:
        score_frame = pd.DataFrame(score_rows)
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        for split_name in ["valid", "test"]:
            values = score_frame[(score_frame["split"] == split_name) & (score_frame["is_accepted"] == True)]["max_reason_prob"]
            axes[0].hist(values, bins=20, alpha=0.5, label=f"{split_name} accepted")
        axes[0].axvline(DEFAULT_ISSUE_THRESHOLD, color="red", linestyle="--", label="issue=0.5")
        axes[0].axvline(DEFAULT_HIGH_ISSUE_THRESHOLD, color="purple", linestyle="--", label="high=0.8")
        axes[0].set_title("Accepted route max reason prob")
        axes[0].legend()

        for split_name in ["valid", "test"]:
            values = score_frame[(score_frame["split"] == split_name) & score_frame["max_labeled_reason_prob"].notna()]["max_labeled_reason_prob"]
            axes[1].hist(values, bins=20, alpha=0.5, label=f"{split_name} rejected labeled")
        axes[1].axvline(DEFAULT_ISSUE_THRESHOLD, color="red", linestyle="--", label="issue=0.5")
        axes[1].set_title("Rejected labeled reason prob")
        axes[1].legend()
        plt.tight_layout()
        plt.show()


## Goodness 阈值分布诊断

用当前模型对 valid/test 中带 accepted/rejected 标签的路线计算 `routeGoodnessProb`，观察实际 choose 与 reject 在 goodness 分数上的重叠程度。


In [ ]:
# Goodness 阈值诊断：先看 accepted/rejected 在 routeGoodnessProb 上是否可分，再讨论 reason。
if "score_rows" not in globals():
    score_rows = current_model_reason_distribution(SPLIT_GROUPS)

if score_rows is None:
    display(Markdown(f"没有找到当前模型：`{OUTPUT_DIR / 'route-preference.pt'}`"))
elif pd is None:
    display(Markdown("需要 pandas 才能生成 goodness 分布表。"))
else:
    score_frame = pd.DataFrame(score_rows)
    labeled_goodness_frame = score_frame[
        score_frame["split"].isin(["valid", "test"])
        & ((score_frame["is_accepted"] == True) | (score_frame["is_rejected"] == True))
    ].copy()
    labeled_goodness_frame["label"] = labeled_goodness_frame["is_accepted"].map({True: "accepted", False: "rejected"})

    if labeled_goodness_frame.empty:
        display(Markdown("valid/test 中没有带 accepted/rejected 的 goodness 标签。"))
    else:
        threshold_rows = []
        for split_name in ["valid", "test"]:
            split_frame = labeled_goodness_frame[labeled_goodness_frame["split"] == split_name]
            if split_frame.empty:
                continue
            for threshold in [0.50, DEFAULT_GOOD_ROUTE_THRESHOLD]:
                pred_accept = split_frame["goodness_prob"] >= threshold
                actual_accept = split_frame["is_accepted"] == True
                tp = int((pred_accept & actual_accept).sum())
                tn = int((~pred_accept & ~actual_accept).sum())
                fp = int((pred_accept & ~actual_accept).sum())
                fn = int((~pred_accept & actual_accept).sum())
                threshold_rows.append(
                    {
                        "split": split_name,
                        "threshold": threshold,
                        "accuracy": round((tp + tn) / len(split_frame), 4),
                        "accepted_recall": round(tp / max(tp + fn, 1), 4),
                        "accepted_precision": round(tp / max(tp + fp, 1), 4),
                        "rejected_recall": round(tn / max(tn + fp, 1), 4),
                        "accepted_as_reject": fn,
                        "rejected_as_accept": fp,
                        "total": len(split_frame),
                    }
                )
        show_table(threshold_rows, "Goodness 固定阈值表现")

        fig, axes = plt.subplots(1, 2, figsize=(15, 4), sharey=True)
        bins = [index / 20 for index in range(21)]
        colors = {"accepted": "tab:blue", "rejected": "tab:orange"}
        for axis, split_name in zip(axes, ["valid", "test"]):
            split_frame = labeled_goodness_frame[labeled_goodness_frame["split"] == split_name]
            for label in ["accepted", "rejected"]:
                values = split_frame[split_frame["label"] == label]["goodness_prob"]
                axis.hist(values, bins=bins, alpha=0.55, label=f"{split_name} {label}", color=colors[label])
            axis.axvline(0.50, color="red", linestyle="--", label="0.50")
            axis.axvline(DEFAULT_GOOD_ROUTE_THRESHOLD, color="purple", linestyle="--", label=f"default={DEFAULT_GOOD_ROUTE_THRESHOLD:.2f}")
            axis.set_title(f"{split_name} goodness prob")
            axis.set_xlabel("routeGoodnessProb")
            axis.set_ylabel("route count")
            axis.set_xlim(0.0, 1.0)
            axis.legend()
        plt.tight_layout()
        plt.show()

        bin_edges = [index / 20 for index in range(21)]
        bin_labels = [f"[{bin_edges[index]:.2f}, {bin_edges[index + 1]:.2f})" for index in range(len(bin_edges) - 1)]
        bin_labels[-1] = f"[{bin_edges[-2]:.2f}, {bin_edges[-1]:.2f}]"

        def goodness_bin_label(value: float) -> str:
            index = min(max(int(value * 20), 0), len(bin_labels) - 1)
            return bin_labels[index]

        labeled_goodness_frame["goodness_bin"] = labeled_goodness_frame["goodness_prob"].map(goodness_bin_label)
        bin_rows = []
        for split_name in ["valid", "test"]:
            split_frame = labeled_goodness_frame[labeled_goodness_frame["split"] == split_name]
            for bin_label in bin_labels:
                bin_frame = split_frame[split_frame["goodness_bin"].astype(str) == bin_label]
                accepted_count = int((bin_frame["label"] == "accepted").sum())
                rejected_count = int((bin_frame["label"] == "rejected").sum())
                if accepted_count == 0 and rejected_count == 0:
                    continue
                total = accepted_count + rejected_count
                bin_rows.append(
                    {
                        "split": split_name,
                        "goodness_bin": bin_label,
                        "accepted_count": accepted_count,
                        "rejected_count": rejected_count,
                        "accepted_ratio": round(accepted_count / total, 4) if total else 0.0,
                        "total": total,
                    }
                )
        show_table(bin_rows, "Goodness 分箱计数（看默认阈值附近 choose/reject 各多少）")


In [ ]:
# 相邻名次排序诊断：按共识名次差 (gap) 分桶的 pairwise 准确率
# gap = 同组两条路线在共识名次里的距离。gap=1 相邻（最难、最值钱），gap 越大越易分。
def gap_bucketed_pairwise(groups_by_split, max_gap: int = 5):
    checkpoint_path = OUTPUT_DIR / "route-preference.pt"
    if not checkpoint_path.exists():
        display(Markdown(f"没有找到当前模型：`{checkpoint_path}`"))
        return None
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    model = RoutePreferenceModel(model_config_from_checkpoint(checkpoint))
    model.load_state_dict(checkpoint["stateDict"])
    model.eval()

    device = torch.device("cpu")
    stats = {sp: defaultdict(lambda: [0, 0]) for sp in groups_by_split}  # gap -> [correct, total]
    overall = {sp: [0, 0] for sp in groups_by_split}
    with torch.no_grad():
        for split_name, split_groups in groups_by_split.items():
            for group in split_groups:
                batch = next(iter_batches((group,), 1, shuffle=False, seed=SEED, device=device))
                if batch.pair_chosen_index.numel() == 0:
                    continue
                output = model(
                    batch.stop_matrix,
                    batch.segment_matrix,
                    batch.route_derived_vector,
                    batch.context_cross_vector,
                    batch.intra_set_vector,
                )
                scores = output.route_preference_score
                # 单组 batch：pair 索引即组内共识名次（items 已按共识名次排序），gap = |名次差|
                for chosen_idx, rejected_idx in zip(
                    batch.pair_chosen_index.tolist(), batch.pair_rejected_index.tolist()
                ):
                    correct = 1 if scores[chosen_idx] > scores[rejected_idx] else 0
                    gap = abs(rejected_idx - chosen_idx)
                    stats[split_name][gap][0] += correct
                    stats[split_name][gap][1] += 1
                    overall[split_name][0] += correct
                    overall[split_name][1] += 1
    return stats, overall, max_gap


_gap_result = gap_bucketed_pairwise(SPLIT_GROUPS, max_gap=5)
if _gap_result is not None:
    gap_stats, gap_overall, MAX_GAP = _gap_result
    gaps = list(range(1, MAX_GAP + 1))
    splits_order = [name for name in ["train", "valid", "test"] if name in gap_stats]

    table_rows = []
    for split_name in splits_order:
        row = {"split": split_name}
        correct_total, total = gap_overall[split_name]
        row["overall"] = round(correct_total / total, 4) if total else None
        for gap in gaps:
            correct, count = gap_stats[split_name].get(gap, [0, 0])
            row[f"gap={gap}"] = f"{correct / count:.3f} (n={count})" if count else "—"
        table_rows.append(row)
    show_table(table_rows, "按共识名次差分桶的 pairwise 准确率（gap=1 最相邻、最难）")

    fig, ax = plt.subplots(figsize=(11, 4.5))
    width = 0.26
    positions = list(range(len(gaps)))
    for series_index, split_name in enumerate(splits_order):
        accuracies = []
        for gap in gaps:
            correct, count = gap_stats[split_name].get(gap, [0, 0])
            accuracies.append(correct / count if count else 0.0)
        offsets = [pos + (series_index - (len(splits_order) - 1) / 2) * width for pos in positions]
        bars = ax.bar(offsets, accuracies, width=width, label=split_name)
        for rect, gap in zip(bars, gaps):
            count = gap_stats[split_name].get(gap, [0, 0])[1]
            if count:
                ax.annotate(
                    f"{rect.get_height():.2f}",
                    (rect.get_x() + rect.get_width() / 2, rect.get_height()),
                    ha="center", va="bottom", fontsize=7,
                )
    ax.axhline(0.5, color="gray", linestyle="--", alpha=0.7, label="随机=0.5")
    ax.set_xticks(positions)
    ax.set_xticklabels([f"gap={gap}" for gap in gaps])
    ax.set_ylim(0.4, 1.0)
    ax.set_ylabel("pairwise 准确率")
    ax.set_title("相邻(gap=1)最难、远(gap大)易分；用于定位排序瓶颈")
    ax.legend()
    plt.tight_layout()
    plt.show()


## gap=1 margin / tie-band 诊断

这一段判断相邻 pair 错误是低置信边界问题，还是模型高置信判断反了。


In [ ]:
# gap=1 margin / tie-band 诊断：判断相邻错误是低置信边界，还是高置信错判。
def collect_margin_diagnostics(groups_by_split):
    checkpoint_path = OUTPUT_DIR / "route-preference.pt"
    if not checkpoint_path.exists():
        display(Markdown(f"没有找到当前模型：`{checkpoint_path}`"))
        return None, None
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    model = RoutePreferenceModel(model_config_from_checkpoint(checkpoint))
    model.load_state_dict(checkpoint["stateDict"])
    model.eval()

    pair_rows = []
    top_rows = []
    device = torch.device("cpu")
    with torch.no_grad():
        for split_name, split_groups in groups_by_split.items():
            for group in split_groups:
                batch = next(iter_batches((group,), 1, shuffle=False, seed=SEED, device=device))
                output = model(
                    batch.stop_matrix,
                    batch.segment_matrix,
                    batch.route_derived_vector,
                    batch.context_cross_vector,
                    batch.intra_set_vector,
                )
                scores = output.route_preference_score.cpu().tolist()
                route_codes = [item.route_code for item in group.items]
                for pair in group.pairs:
                    margin = scores[pair.chosen_index] - scores[pair.rejected_index]
                    gap = abs(pair.rejected_index - pair.chosen_index)
                    pair_rows.append(
                        {
                            "split": split_name,
                            "candidate_set_id": group.candidate_set_id,
                            "chosen_route_code": pair.chosen_route_code,
                            "rejected_route_code": pair.rejected_route_code,
                            "gap": gap,
                            "gap_bucket": f"gap={gap}" if gap <= 3 else "gap>=4",
                            "score_chosen": scores[pair.chosen_index],
                            "score_rejected": scores[pair.rejected_index],
                            "margin": margin,
                            "abs_margin": abs(margin),
                            "correct": margin > 0,
                            "pair_weight": pair.weight_raw,
                        }
                    )
                if len(scores) >= 2:
                    predicted = sorted(range(len(scores)), key=lambda index: scores[index], reverse=True)
                    top1_index = predicted[0]
                    top2_index = predicted[1]
                    top_margin = scores[top1_index] - scores[top2_index]
                    top_rows.append(
                        {
                            "split": split_name,
                            "candidate_set_id": group.candidate_set_id,
                            "pred_top1_route_code": route_codes[top1_index],
                            "pred_top2_route_code": route_codes[top2_index],
                            "label_top1_route_code": next(item.route_code for item in group.items if item.rank == 0),
                            "top1_margin": top_margin,
                            "top1_correct": group.items[top1_index].rank == 0,
                            "top2_hit": any(group.items[index].rank == 0 for index in predicted[:2]),
                        }
                    )
    return pair_rows, top_rows


def _pct(value: float) -> float:
    return round(value * 100, 2)


def summarize_margin_by_gap(pair_rows, split_names=("valid", "test")):
    rows = []
    for split_name in split_names:
        split_rows = [row for row in pair_rows if row["split"] == split_name]
        for gap_bucket in ["gap=1", "gap=2", "gap=3", "gap>=4"]:
            bucket_rows = [row for row in split_rows if row["gap_bucket"] == gap_bucket]
            if not bucket_rows:
                continue
            correct_rows = [row for row in bucket_rows if row["correct"]]
            wrong_rows = [row for row in bucket_rows if not row["correct"]]
            rows.append(
                {
                    "split": split_name,
                    "gap": gap_bucket,
                    "pairs": len(bucket_rows),
                    "accuracy": _pct(len(correct_rows) / len(bucket_rows)),
                    "abs_margin_p50": round(percentile([row["abs_margin"] for row in bucket_rows], 0.50), 4),
                    "abs_margin_p90": round(percentile([row["abs_margin"] for row in bucket_rows], 0.90), 4),
                    "correct_margin_p50": round(percentile([row["margin"] for row in correct_rows], 0.50), 4),
                    "wrong_abs_margin_p50": round(percentile([row["abs_margin"] for row in wrong_rows], 0.50), 4),
                    "wrong_abs_margin_p90": round(percentile([row["abs_margin"] for row in wrong_rows], 0.90), 4),
                }
            )
    return rows


def tie_band_rows(pair_rows, split_names=("valid", "test"), gap_bucket="gap=1"):
    thresholds = [0.02, 0.05, 0.10, 0.15, 0.20, 0.30]
    rows = []
    for split_name in split_names:
        base_rows = [row for row in pair_rows if row["split"] == split_name and row["gap_bucket"] == gap_bucket]
        wrong_rows = [row for row in base_rows if not row["correct"]]
        for threshold in thresholds:
            abstained = [row for row in base_rows if row["abs_margin"] < threshold]
            remaining = [row for row in base_rows if row["abs_margin"] >= threshold]
            remaining_correct = [row for row in remaining if row["correct"]]
            covered_wrong = [row for row in wrong_rows if row["abs_margin"] < threshold]
            rows.append(
                {
                    "split": split_name,
                    "gap": gap_bucket,
                    "tie_threshold": threshold,
                    "abstain_rate": _pct(len(abstained) / len(base_rows)) if base_rows else 0.0,
                    "wrong_covered_rate": _pct(len(covered_wrong) / len(wrong_rows)) if wrong_rows else 0.0,
                    "remaining_pairs": len(remaining),
                    "remaining_accuracy": _pct(len(remaining_correct) / len(remaining)) if remaining else 0.0,
                }
            )
    return rows


def top_margin_tie_rows(top_rows, split_names=("valid", "test")):
    thresholds = [0.02, 0.05, 0.10, 0.15, 0.20, 0.30]
    rows = []
    for split_name in split_names:
        split_rows = [row for row in top_rows if row["split"] == split_name]
        top1_wrong_top2_hit = [row for row in split_rows if not row["top1_correct"] and row["top2_hit"]]
        for threshold in thresholds:
            low_margin = [row for row in split_rows if row["top1_margin"] < threshold]
            covered_top1_flip = [row for row in top1_wrong_top2_hit if row["top1_margin"] < threshold]
            remaining = [row for row in split_rows if row["top1_margin"] >= threshold]
            rows.append(
                {
                    "split": split_name,
                    "top1_top2_threshold": threshold,
                    "low_margin_set_rate": _pct(len(low_margin) / len(split_rows)) if split_rows else 0.0,
                    "top1_flip_covered_rate": _pct(len(covered_top1_flip) / len(top1_wrong_top2_hit)) if top1_wrong_top2_hit else 0.0,
                    "remaining_sets": len(remaining),
                    "remaining_top1_accuracy": _pct(sum(1 for row in remaining if row["top1_correct"]) / len(remaining)) if remaining else 0.0,
                }
            )
    return rows


pair_margin_rows, top_margin_rows = collect_margin_diagnostics(SPLIT_GROUPS)
if pair_margin_rows is not None:
    show_table(summarize_margin_by_gap(pair_margin_rows), "按 gap 的模型 score margin 分布")
    show_table(tie_band_rows(pair_margin_rows), "gap=1 pair tie-band 覆盖率")
    show_table(top_margin_tie_rows(top_margin_rows), "top1/top2 margin tie-band 覆盖率")

    high_conf_wrong_rows = [
        row for row in pair_margin_rows
        if row["split"] in {"valid", "test"} and row["gap_bucket"] == "gap=1" and not row["correct"]
    ]
    high_conf_wrong_rows = sorted(high_conf_wrong_rows, key=lambda row: row["abs_margin"], reverse=True)[:20]
    show_table(
        [
            {
                "split": row["split"],
                "candidate_set_id": row["candidate_set_id"],
                "chosen_route_code": row["chosen_route_code"],
                "rejected_route_code": row["rejected_route_code"],
                "margin": round(row["margin"], 4),
                "abs_margin": round(row["abs_margin"], 4),
                "pair_weight": round(row["pair_weight"], 4),
            }
            for row in high_conf_wrong_rows
        ],
        "valid/test gap=1 高置信错样本 Top 20",
    )

    if pd is not None:
        margin_frame = pd.DataFrame(pair_margin_rows)
        focus_frame = margin_frame[margin_frame["split"].isin(["valid", "test"])]
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        for split_name, axis in zip(["valid", "test"], axes):
            gap1 = focus_frame[(focus_frame["split"] == split_name) & (focus_frame["gap_bucket"] == "gap=1")]
            if gap1.empty:
                continue
            axis.hist(gap1[gap1["correct"]]["abs_margin"], bins=30, alpha=0.65, label="correct")
            axis.hist(gap1[~gap1["correct"]]["abs_margin"], bins=30, alpha=0.65, label="wrong")
            axis.set_title(f"{split_name} gap=1 abs margin")
            axis.set_xlabel("abs(score margin)")
            axis.set_ylabel("pairs")
            axis.legend()
        plt.tight_layout()
        plt.show()


## 如何解读

- 如果 train split 的某个 reason code 正样本少于 30，优先补该 code 的数据，不要靠调参硬救。
- 如果 candidate set 数量很少，优先补新的 candidate set；route 数多但 candidate set 少，泛化仍然不稳。
- 如果 train 指标继续涨、valid 指标早早见顶，优先使用 `best_metric` / early stopping，而不是继续加大模型。
- `acceptedReasonFalsePositiveRate@issueThreshold` 使用 0.5 阈值，是偏严格的诊断；实际预测对 good route 会用 `HIGH_ISSUE_THRESHOLD=0.8`。
